# Linear regression with scikit-learn 📈📈

We work with a small dataset of customers (country, age, salary, and whether they purchased a product). The objective: **predict the salary** from the other variables.

To do so, we'll train a linear regression model and evaluate its performances. The preprocessing is similar to the one of the preprocessing lessons (without label encoding: our target is already numeric). The dataset is small and simple, so you can focus on modelling and performance evaluation. We also added a few figures, because EDA is an important step before training any model : it allows to understand better the relationship between the different variables in the dataset 🤓

## What will you learn in this course? 🧐🧐

This lecture is a follow-along demo to walk you through building a linear regression model in practice using sklearn!

* EDA
* Baseline model : simple linear regression
    * Training pipeline
    * Test pipeline
* Multivariate linear regression
    * Training pipeline
    * Test pipeline


## EDA

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"


In [ ]:
# Import dataset
print("Loading dataset...")
dataset = pd.read_csv("data/customers.csv")
print("...Done.")
print()

Loading dataset...
...Done.



In [ ]:
# Basic stats
data_desc = dataset.describe(include='all')
print(dataset.shape)
data_desc

(400, 4)


,Country,Age,Salary,Purchased
count,400,391.000000,400.000000,400
unique,4,NaN,NaN,2
top,France,NaN,NaN,No
freq,139,NaN,NaN,241
mean,NaN,42.437340,47013.500000,NaN
std,NaN,13.297408,14079.992301,NaN
min,NaN,20.000000,13000.000000,NaN
25%,NaN,31.000000,35800.000000,NaN
50%,NaN,43.000000,47250.000000,NaN
75%,NaN,54.000000,58150.000000,NaN


- Taget variable : Y = 'Salary'
- Explanatory variables : 'Country', 'Age', 'Purchased'
- Number of examples : 400

- Imputation of missing values : 'Age'
- Normalisation : 'Age'
- OHE : 'Country', 'Purchased'

In [ ]:
# Univariate analysis
# Distribution of each numeric variable
num_features = ['Age', 'Salary']
for i in range(len(num_features)):
    fig = px.histogram(dataset[num_features[i]])
    fig.show()

In [ ]:
# Univariate analysis
# Barplot of each qualitative variable
cat_features = ['Country', 'Purchased']
for i in range(len(cat_features)):
    fig = px.histogram(dataset, x=cat_features[i])
    fig.show()

In [ ]:
# Correlation matrix
corr_matrix = dataset.corr(numeric_only=True).round(2)

fig = px.imshow(corr_matrix, text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Correlation matrix")
fig.show()

There's a very strong correlation between the target and the Age variable ! This is a good news, it means that a linear model relating these two variables may be a good estimator of the salary 🙂

Let's analyze pairwise scatterplots to investigate the interdependencies between the variables :

In [ ]:
# Visualize pairwise dependencies
fig = px.scatter_matrix(dataset)
fig.update_layout(
        title = go.layout.Title(text = "Bivariate analysis", x = 0.5), showlegend = False, 
            autosize=False, height=800, width = 800)
fig.show()

## Baseline model : simple linear regression
Let's try a first basic model : simple linear regression with only one feature. We choose the age because we just noticed that it is strongly correlated to the salary.

In [ ]:
# Separate target variable Y from features X
print("Separating labels from features...")
features_list = ["Age"]
target_variable = "Salary"

X = dataset.loc[:,features_list]
Y = dataset.loc[:,target_variable]

print("...Done.")
print()

print('Y : ')
print(Y.head())
print()
print('X :')
print(X.head())

Separating labels from features...
...Done.

Y : 
0    38500.0
1    32200.0
2    47900.0
3    54300.0
4    51800.0
Name: Salary, dtype: float64

X :
    Age
0  20.0
1  33.0
2  43.0
3  62.0
4  53.0


In [ ]:
# Divide dataset Train set & Test set 
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
print("...Done.")
print()

Dividing into train and test sets...
...Done.



### Preprocessing
:::note Simplified syntax
In this first example, we don't need to use the class `ColumnTransformer` because the preprocessing is very simple : it just consists in one step of imputation followed by a standardization, and these steps will be applied to the whole DataFrame (as there's actually only one column!). In this case, we can just create a `Pipeline` that describes the different steps and call the `fit_transform` and `transform` methods directly on it, by passing X_train/X_test as argument.
:::

In [ ]:
print("Preprocessing X_train...")
print(X_train.head())
print()
preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
X_train = preprocessor.fit_transform(X_train)
print("...Done!")
print(X_train[0:5,:]) # X_train is now a numpy array
print() 

Preprocessing X_train...
      Age
336  53.0
64   58.0
55   59.0
106  38.0
300  42.0

...Done!
[[ 0.77143177]
 [ 1.14762526]
 [ 1.22286396]
 [-0.35714869]
 [-0.0561939 ]]



In [ ]:
# Test pipeline
print("Preprocessing X_test...")
print(X_test.head())
print()
X_test = preprocessor.transform(X_test)
print("...Done!")
print(X_test[0:5,:]) # X_test is now a numpy array
print() 

Preprocessing X_test...
      Age
132  31.0
309  22.0
341  35.0
196  44.0
246  31.0

...Done!
[[-0.88381958]
 [-1.56096786]
 [-0.58286479]
 [ 0.09428349]
 [-0.88381958]]



### Train model

In [ ]:
# Train model
print("Train model...")
regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("...Done.")

Train model...
...Done.


### Performance assessment

In [ ]:
# Predictions on training set
print("Predictions on training set...")
Y_train_pred = regressor.predict(X_train)
print("...Done.")
print(Y_train_pred)
print()

Predictions on training set...
...Done.
[56352.31180689 60895.80430645 61804.50280636 42721.83430824
 46356.62830788 39995.73880851 44539.23130806 33634.84930913
 52717.51780725 60895.80430645 50900.12080743 66347.99530591
 36360.94480887 41813.13580833 39995.73880851 66347.99530591
 65439.296806   43630.53280815 57261.0103068  47265.32680779
 47265.32680779 58169.70880671 33634.84930913 42721.83430824
 49082.72380761 62713.20130627 61804.50280636 61804.50280636
 43630.53280815 29091.35680958 47265.32680779 37269.64330878
 60895.80430645 53626.21630716 30000.05530949 41813.13580833
 39087.0403086  50900.12080743 44539.23130806 60895.80430645
 48174.0253077  60895.80430645 29091.35680958 56352.31180689
 33634.84930913 64530.59830609 47265.32680779 47265.32680779
 41813.13580833 64530.59830609 30000.05530949 55443.61330698
 59987.10580653 55443.61330698 27273.95980976 27273.95980976
 35452.24630896 48174.0253077  61804.50280636 30000.05530949
 65439.296806   51808.81930734 36360.94480887

In [ ]:
# Predictions on test set
print("Predictions on test set...")
Y_test_pred = regressor.predict(X_test)
print("...Done.")
print(Y_test_pred)
print()

Predictions on test set...
...Done.
[36360.94480887 28182.65830967 39995.73880851 48174.0253077
 36360.94480887 66347.99530591 51808.81930734 54534.91480707
 42721.83430824 47265.32680779 34543.54780904 44539.23130806
 67256.69380582 54534.91480707 54534.91480707 59078.40730662
 42721.83430824 54534.91480707 47265.32680779 47265.32680779
 30000.05530949 39995.73880851 36360.94480887 39995.73880851
 35452.24630896 38178.34180869 49082.72380761 60895.80430645
 59078.40730662 35452.24630896 38178.34180869 31817.45230931
 51808.81930734 28182.65830967 55443.61330698 48174.0253077
 30000.05530949 45447.92980797 49991.42230752 51808.81930734
 26365.26130985 37269.64330878 49082.72380761 38178.34180869
 57261.0103068  38178.34180869 26365.26130985 67256.69380582
 66347.99530591 39087.0403086  50900.12080743 30908.7538094
 39995.73880851 43630.53280815 64530.59830609 47265.32680779
 66347.99530591 28182.65830967 54534.91480707 26365.26130985
 55443.61330698 57261.0103068  56352.31180689 39087.

In [ ]:
# Print R^2 scores
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))

R2 score on training set :  0.7217600554604138
R2 score on test set :  0.704882350693096


In [ ]:
# Visualize the model
# Visualize predictions on training Set
fig = px.scatter(x = X_train.flatten().tolist(), y = Y_train, title = "training set")
fig.add_trace(go.Scatter(x = X_train.flatten().tolist(), y = Y_train_pred, name = "linear regression"))
fig.show()

# Visualize predictions on test Set
fig = px.scatter(x = X_test.flatten().tolist(), y = Y_test, title = "test set")
fig.add_trace(go.Scatter(x = X_test.flatten().tolist(), y = Y_test_pred, name = "linear regression"))
fig.show()

In the above example all assumptions needed to train a simple linear regression model are verified.

* There seems to be a linear relation between $Y$ and the feature $X$
* The deviation between the prediction and the actual values of $Y$, i.e. the residuals, seem to be comparable accross the values of $Y$ (this is Homoscedaticity).
* There does not seem to be any autocorrelation going on with the residuals, they seem to be randomly distributed accross all values of $Y$

In addition to that, our model seems to be a good predictor based on our training samples.

## Multivariate linear regression
Let's train a multivariate model by adding the categorical features : Country and Purchased.

In [ ]:
# Separate target variable Y from features X
print("Separating labels from features...")
features_list = ["Country", "Age", "Purchased"]
target_variable = "Salary"

X = dataset.loc[:,features_list]
Y = dataset.loc[:,target_variable]

print("...Done.")
print()

print('Y : ')
print(Y.head())
print()
print('X :')
print(X.head())

Separating labels from features...
...Done.

Y : 
0    38500.0
1    32200.0
2    47900.0
3    54300.0
4    51800.0
Name: Salary, dtype: float64

X :
   Country   Age Purchased
0  Germany  20.0        No
1    Spain  33.0        No
2    Italy  43.0        No
3  Germany  62.0        No
4   France  53.0       Yes


In [ ]:
# Automatically detect names of numeric/categorical columns
numeric_features = []
categorical_features = []
for i,t in X.dtypes.items():
    if ('float' in str(t)) or ('int' in str(t)) :
        numeric_features.append(i)
    else :
        categorical_features.append(i)

print('Found numeric features ', numeric_features)
print('Found categorical features ', categorical_features)

Found numeric features  ['Age']
Found categorical features  ['Country', 'Purchased']


In [ ]:
# Divide dataset Train set & Test set 
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
print("...Done.")
print()

Dividing into train and test sets...
...Done.



### Preprocessing

In [ ]:
# Create pipeline for numeric features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')), # missing values will be replaced by columns' mean
    ('scaler', StandardScaler())
])

In [ ]:
# Create pipeline for categorical features
categorical_transformer = Pipeline(
    steps=[
    ('encoder', OneHotEncoder(drop='first')) # first column will be dropped to avoid creating correlations between features
    ])

In [ ]:
# Use ColumnTransformer to make a preprocessor object that describes all the treatments to be done
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [ ]:
# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train = preprocessor.fit_transform(X_train)
print('...Done.')
print(X_train[0:5]) # MUST use this syntax because X_train is a numpy array and not a pandas DataFrame anymore
print()

# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head()) 
X_test = preprocessor.transform(X_test) # Don't fit again !! The test set is used for validating decisions
# we made based on the training set, therefore we can only apply transformations that were parametered using the training set.
# Otherwise this creates what is called a leak from the test set which will introduce a bias in all your results.
print('...Done.')
print(X_test[0:5,:]) # MUST use this syntax because X_test is a numpy array and not a pandas DataFrame anymore
print()

Performing preprocessings on train set...
     Country   Age Purchased
336   France  53.0       Yes
64   Germany  58.0       Yes
55   Germany  59.0        No
106    Spain  38.0        No
300  Germany  42.0        No
...Done.
[[ 0.77179836  0.          0.          0.          1.        ]
 [ 1.14799315  1.          0.          0.          1.        ]
 [ 1.22323211  1.          0.          0.          0.        ]
 [-0.35678602  0.          0.          1.          0.        ]
 [-0.05583018  1.          0.          0.          0.        ]]

Performing preprocessings on test set...
     Country   Age Purchased
132    Spain  31.0        No
309   France  22.0        No
341    Italy  35.0        No
196    Spain  44.0        No
246  Germany  31.0        No
...Done.
[[-0.88345873  0.          0.          1.          0.        ]
 [-1.56060935  0.          0.          0.          0.        ]
 [-0.58250289  0.          1.          0.          0.        ]
 [ 0.09464773  0.          0.          1.    

### Train model

In [ ]:
# Train model
print("Train model...")
regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("...Done.")

Train model...
...Done.


### Performance assessment

In [ ]:
# Predictions on training set
print("Predictions on training set...")
Y_train_pred = regressor.predict(X_train)
print("...Done.")
print(Y_train_pred)
print()

Predictions on training set...
...Done.
[59239.15259381 65431.29572812 62839.63738915 36144.65485822
 49149.43117015 41346.6755395  48770.17136752 35709.53180226
 48399.9881753  65431.29572812 43392.41109181 68097.52132375
 30507.51112098 46354.25262299 37372.84126677 68097.52132375
 67292.21507558 46733.51242562 56070.62456926 49954.73741833
 53351.70200547 59618.41239644 39106.49638941 49325.17076459
 49399.73802126 64876.29633105 56453.05065707 64070.99008287
 36949.9611064  27709.16628865 43568.15068624 31312.81736916
 59291.84956197 45808.32983634 24870.36738375 42957.28803585
 32923.42986551 51010.35051761 45373.20678038 55647.74440889
 40976.49234728 59291.84956197 27709.16628865 55265.31832108
 28091.59237646 62513.07455467 43607.55301618 40171.1860991
 45122.89992927 66486.9088274  34653.91870298 60599.45823906
 54842.43816071 60599.45823906 26098.5537923  32237.99995845
 29702.20487281 51991.39636023 66236.60197629 32488.30680956
 59674.27564977 47594.68192713 40291.06244021 

In [ ]:
# Predictions on test set
print("Predictions on test set...")
Y_test_pred = regressor.predict(X_test)
print("...Done.")
print(Y_test_pred)
print()

Predictions on test set...
...Done.
[30507.51112098 30877.6943132  37372.84126677 40976.49234728
 40291.06244021 70263.13321717 53981.26865921 59794.15199088
 36144.65485822 53351.70200547 28896.89862463 37755.26735457
 61284.88814612 50010.60067165 56397.18740374 61655.07133834
 43762.59428403 50010.60067165 50978.35187606 47789.12552491
 24870.36738375 41346.6755395  38125.45054679 43512.28743292
 37320.14429862 32118.12361734 41781.79859545 63265.6838347
 61655.07133834 37320.14429862 39736.06304314 29877.94446725
 51815.65676579 26903.86004047 50815.90691983 48594.43177308
 35885.2713967  45601.64334297 52370.65616286 44197.71733998
 25293.24754412 38930.75679497 51565.34991468 32118.12361734
 58813.10614827 41901.67493656 29267.08181685 71068.43946535
 70263.13321717 32923.42986551 43392.41109181 25675.67363193
 43512.28743292 36949.9611064  66486.9088274  50978.35187606
 70263.13321717 23259.7548874  50010.60067165 31432.69371027
 55036.8817585  60044.45884199 59239.15259381 3292

In [ ]:
# Print R^2 scores
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))

R2 score on training set :  0.8058760696786484
R2 score on test set :  0.7356127353573365


### Conclusion about the model's quality
Here we can see that the R2-score has improved when we went from simple to multiple linear regression (from 0.70 to 0.74 on the test set). This is because we added some features that were indeed useful to predict the values of the target variables. 
Sometimes, adding more features won't necessarily improve the performances, and it can even deteriorate in some cases ! That's why in general, some feature selection methods are required to choose the right set of variables to be included into the model.

:::note What about plotting ?
You can notice we didn't plot the multivariate model. It's simply because we can't ! To make a graphical representation of this model, we would need 4 dimensions (3 features + the target). Unfortunately, our brain is able to visualize only 3 at a time 🥴
This is the reason why it's so important to always compute some metrics such as the R2-score to assess the quality of our predictions: most of the times, we're "blind" on what's happening and we have to rely on these scores to make sure the model is making good predictions !
:::

### Interpreting the model's coefficients
As we've standardized our features, we can use the coefficients of the regression to estimate the importance of each feature for the prediction. The model's parameters are saved in a `.coef_` attribute:

In [ ]:
regressor.coef_

array([10703.31466314,  2165.61189342, -3973.83427273, -7617.93942581,
        3396.96458714])

Each coefficient can be linked with the name of the corresponding feature by digging into the different pipelines that were used to produce the final version of the X_train/X_test arrays:

In [ ]:
column_names = []
for name, pipeline, features_list in preprocessor.transformers_: # loop over pipelines
    if name == 'num': # if pipeline is for numeric variables
        features = features_list # just get the names of columns to which it has been applied
    else: # if pipeline is for categorical variables
        features = pipeline.named_steps['encoder'].get_feature_names_out() # get output columns names from OneHotEncoder
    column_names.extend(features) # concatenate features names
        
print("Names of columns corresponding to each coefficient: ", column_names)

Names of columns corresponding to each coefficient:  ['Age', 'Country_Germany', 'Country_Italy', 'Country_Spain', 'Purchased_Yes']


In [ ]:
# Create a pandas DataFrame
coefs = pd.DataFrame(index = column_names, data = regressor.coef_.transpose(), columns=["coefficients"])
coefs

,coefficients
Age,10703.314663
Country_Germany,2165.611893
Country_Italy,-3973.834273
Country_Spain,-7617.939426
Purchased_Yes,3396.964587


:::warning Comparison in absolute values
The feature importance is related to the absolute values of the coefficients. Don't forget to compute the absolute value before concluding !
:::

In [ ]:
# Compute abs() and sort values
feature_importance = abs(coefs).sort_values(by = 'coefficients')
feature_importance

,coefficients
Country_Germany,2165.611893
Purchased_Yes,3396.964587
Country_Italy,3973.834273
Country_Spain,7617.939426
Age,10703.314663


In [ ]:
# Plot coefficients
fig = px.bar(feature_importance, orientation = 'h')
fig.update_layout(showlegend = False, 
                  margin = {'l': 120} # to avoid cropping of column names
                 )
fig.show()